# NIAT Demo: AI Prompt-to-Image and Image Editing

Run this on Google Colab with a FREE GPU!

### Before you start:
1. Go to **Runtime - Change runtime type - T4 GPU** - Save
2. Run each cell top-to-bottom with Shift+Enter
3. A public Gradio link appears at the end - click it!


## Step 1: Check GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device.upper())
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU ready! Images generate in ~15-30 seconds.')
else:
    print('No GPU detected. Go to Runtime - Change runtime type - T4 GPU')


## Step 2: Install Dependencies (takes 2-3 min)

In [ ]:
print('Installing...')
import subprocess
subprocess.run(['pip', 'install', '-q', 'gradio', 'diffusers', 'transformers',
                'accelerate', 'sentencepiece', 'huggingface_hub', 'Pillow'])
print('Done!')


## Step 3: Load Qwen LLM

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32
LLM_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

print('Loading Qwen on', device.upper(), '...')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, torch_dtype=dtype,
    device_map='auto' if device == 'cuda' else None
)
if device == 'cpu':
    llm_model = llm_model.to('cpu')
print('Qwen loaded OK')

def _llm(messages):
    txt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(txt, return_tensors='pt').to(llm_model.device)
    out = llm_model.generate(
        **inp, max_new_tokens=256, temperature=0.7,
        do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0][inp['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

SYS_GEN = 'You are an expert prompt engineer for AI image generation. Rewrite the prompt to be more detailed and vivid. Output only the improved prompt.'
SYS_EDIT = 'You are an expert prompt engineer for AI image editing. Rewrite the edit instruction to be clear and specific. Output only the improved prompt.'

def engineer_generation_prompt(p):
    return _llm([{'role':'system','content':SYS_GEN},{'role':'user','content':p}])

def engineer_editing_prompt(p):
    return _llm([{'role':'system','content':SYS_EDIT},{'role':'user','content':p}])

# Quick test
test = engineer_generation_prompt('a cat eating fish')
print('Test prompt:', test[:100])


## Step 4: Load Stable Diffusion (downloads ~4GB on first run)

In [ ]:
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline

SD_MODEL = 'runwayml/stable-diffusion-v1-5'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading Stable Diffusion on', device.upper(), '...')
print('This downloads ~4GB on first run. Please wait...')

if device == 'cuda':
    txt2img = StableDiffusionPipeline.from_pretrained(
        SD_MODEL, torch_dtype=torch.float16, variant='fp16'
    ).to('cuda')
else:
    txt2img = StableDiffusionPipeline.from_pretrained(
        SD_MODEL, torch_dtype=torch.float32
    ).to('cpu')
    txt2img.enable_attention_slicing()

img2img = StableDiffusionImg2ImgPipeline(**txt2img.components)

STEPS = 30 if device == 'cuda' else 6
print('Loaded OK! Steps =', STEPS)


## Step 5: Launch the App

A public link will appear below. Click it to open the app!

In [ ]:
import gradio as gr
from PIL import Image

def generate_fn(prompt):
    if not prompt.strip():
        return 'Please enter a prompt!', None, None
    try:
        print('Enhancing prompt...')
        eng = engineer_generation_prompt(prompt)
        print('Generating image...')
        img = txt2img(eng, num_inference_steps=STEPS, guidance_scale=7.5).images[0]
        print('Done!')
        return eng, img, img
    except Exception as e:
        return 'ERROR: ' + str(e), None, None

def edit_fn(image, instruction):
    if image is None:
        return 'No image to edit!', None, None
    if not instruction.strip():
        return 'Please enter an instruction!', None, None
    try:
        eng = engineer_editing_prompt(instruction)
        resized = image.convert('RGB').resize((512, 512))
        img = img2img(prompt=eng, image=resized, strength=0.7, guidance_scale=8.0, num_inference_steps=STEPS).images[0]
        return eng, img, img
    except Exception as e:
        return 'ERROR: ' + str(e), None, None

with gr.Blocks(title='NIAT: AI Image Generation') as demo:
    gr.Markdown('# NIAT: AI Prompt-to-Image and Image Editing')
    gr.Markdown('Powered by Stable Diffusion + Qwen LLM Prompt Engineering')

    with gr.Tabs():
        with gr.TabItem('Generate Image'):
            with gr.Row():
                with gr.Column():
                    prompt_in = gr.Textbox(label='Your Prompt', placeholder='a cat eating fish', lines=4)
                    gen_btn   = gr.Button('Generate Image', variant='primary')
                    eng_out   = gr.Textbox(label='AI-Enhanced Prompt (Qwen output)', interactive=False, lines=4)
                with gr.Column():
                    img_out = gr.Image(label='Generated Image', type='pil', height=400)
            state = gr.State()
            gen_btn.click(generate_fn, inputs=prompt_in, outputs=[eng_out, img_out, state])

        with gr.TabItem('Edit Image'):
            with gr.Row():
                with gr.Column():
                    upload_in  = gr.Image(label='Upload Image', type='pil', height=250)
                    upload_btn = gr.Button('Use this Image')
                    edit_in    = gr.Textbox(label='Edit Instruction', placeholder='make the cat wear sunglasses', lines=3)
                    edit_btn   = gr.Button('Edit Image', variant='primary')
                    eng_edit   = gr.Textbox(label='AI-Enhanced Edit Prompt', interactive=False, lines=3)
                with gr.Column():
                    orig_out   = gr.Image(label='Original Image', type='pil', height=250)
                    edited_out = gr.Image(label='Edited Image', type='pil', height=250)
            estate = gr.State()
            upload_btn.click(lambda x: (x, x), inputs=upload_in, outputs=[orig_out, estate])
            edit_btn.click(edit_fn, inputs=[estate, edit_in], outputs=[eng_edit, edited_out, orig_out])

demo.queue(max_size=5)
demo.launch(share=True)


## Tips

- Add style: `photorealistic`, `oil painting`, `anime style`, `watercolor`
- Add quality: `highly detailed`, `8k`, `masterpiece`, `sharp focus`
- Example: `a cat eating fish, photorealistic, soft lighting, highly detailed`
- To restart after session expires: **Runtime - Run all**
